In [1]:
!pip install evaluate

In [3]:
import os

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import evaluate


train_path = "../data/processed/train_en_hi_ru_12349.csv"
val_path = "../data/processed/val_en_hi_ru_2403.csv"



# model_name = "google-bert/bert-base-multilingual-cased"
# output_dir = "./mBERT_toxic_model"

model_name = "xlm-roberta-large"
output_dir = "/content/drive/MyDrive/xlm_roberta_model_saved"
#output_dir = "./xlm_roberta_model"

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Training will run on: {device.upper()}")


dataset = load_dataset('csv', data_files={'train': train_path, 'validation': val_path})

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128)


tokenized_datasets = dataset.map(tokenize_function, batched=True)

id2label = {0: "normal", 1: "toxic"}
label2id = {"normal": 0, "toxic": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
).to(device)


metric_accuracy = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = metric_accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = metric_f1.compute(predictions=predictions, references=labels)["f1"]

    return {
        "accuracy": round(acc, 4),
        "f1-score": round(f1, 4)
    }

training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=16,
    fp16=True,
    gradient_checkpointing=True,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1-score",
    logging_dir="./logs",
    logging_steps=50,
    report_to="none"
)


data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\nStarting fine-tuning ")
trainer.train()

print(f"\nTraining complete, model is saved in: {output_dir}")

Training will run on: CUDA


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOA


Starting fine-tuning 


Epoch,Training Loss,Validation Loss,Accuracy,F1-score
1,No log,0.309730,0.899700,0.896100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import shutil

folder_to_zip = "./xlm_roberta_model"

zip_filename = 'xlm_roberta_toxic_model_finetuned'

shutil.make_archive(zip_filename, 'zip', folder_to_zip)
print(f"Archive '{zip_filename}.zip' created")


files.download(f"{zip_filename}.zip")